In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv(r"C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks.csv")

In [31]:
print("Shape\n", df.shape)

Shape
 (307, 6)


In [32]:
print("Data Types\n", df.dtypes)

Data Types
 id              int64
year              str
state_code      int64
state_name        str
route_length    int64
total_track     int64
dtype: object


In [33]:
df.head(20)

,id,year,state_code,state_name,route_length,total_track
0,0,2012-2013,28,Arunachal Pradesh,5322,9941
1,1,2012-2013,12,Assam,1,3
2,2,2012-2013,18,Bihar,2459,3525
3,3,2012-2013,10,Chhatisgarh,3656,6595
4,4,2012-2013,22,Delhi,1196,2625
5,5,2012-2013,7,Goa,183,699
6,6,2012-2013,30,Gujarat,69,98
7,7,2012-2013,24,Haryana,5257,7388
8,8,2012-2013,6,Himachal Pradesh,1630,2973
9,9,2012-2013,2,Jammu and Kashmir,296,357


In [34]:
print("Missing Values\n", df.isna().sum())

Missing Values
 id              0
year            0
state_code      0
state_name      0
route_length    0
total_track     0
dtype: int64


In [35]:
print("Duplicate Rows\n", df.duplicated().sum())

Duplicate Rows
 0


Since 1 row represents, one state per year so key becomes (state_name, year)

In [36]:
key = ['state_name', 'year']
print("Duplicate Rows based on key\n", df.duplicated(subset=key).sum())



Duplicate Rows based on key
 0


In [37]:
print(df[["route_length", "total_track"]].describe())

       route_length   total_track
count    307.000000    307.000000
mean    2186.609121   3972.934853
std     2317.682900   4143.963374
min        1.000000      3.000000
25%       69.000000    117.500000
50%     1703.000000   3058.000000
75%     3817.000000   6835.000000
max    10324.000000  16063.000000


In [38]:
print("\nUnique state_code :", df["state_code"].nunique())
print("Unique state_name :", df["state_name"].nunique())


Unique state_code : 31
Unique state_name : 31


In [39]:
violations =  df[df["route_length"] > df['total_track']]
print('Rows where route_length is greater than total_track (Not possible):', len(violations))
print(violations)

Rows where route_length is greater than total_track (Not possible): 1
      id       year  state_code   state_name  route_length  total_track
178  178  2017-2018           5  Uttarakhand           341          189


In [40]:
codes_with_multiple_names = df.groupby('state_code')['state_name'].nunique()
print("\nHow many names each codes maps to:\n" , codes_with_multiple_names[codes_with_multiple_names > 1])


How many names each codes maps to:
 state_code
1     2
2     2
6     2
7     2
10    2
12    2
18    2
20    2
22    2
23    2
24    2
28    2
29    2
30    2
32    2
Name: state_name, dtype: int64


In [41]:
grid = df.pivot_table(index='state_name', columns='year', values='state_code', aggfunc='first')
changing = grid[grid.nunique(axis=1) > 1]

In [42]:
pd.set_option('display.width', 200, 'display.max_columns', 20)
print(changing)

year               2012-2013  2013-2014  2014-2015  2015-2016  2016-2017  2017-2018  2018-2019  2019-2020  2020-2021  2021-2022
state_name                                                                                                                     
Andhra Pradesh          23.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0       28.0
Arunachal Pradesh       28.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0       12.0
Assam                   12.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0       18.0
Bihar                   18.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0       10.0
Chhatisgarh             10.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0       22.0
Delhi                   22.0        7.0        7.0        7.0        7.0        7.0        7.0        7.

In [44]:
good_year = df[df["year"] != "2012-2013"]
canonical_map = good_year.drop_duplicates("state_name").set_index("state_name")["state_code"].to_dict()
print("Canonical map size:", len(canonical_map), "states")

Canonical map size: 31 states


In [45]:
df["state_code"] = df["state_name"].map(canonical_map)

In [46]:
print("Codes mapping to >1 name:", (df.groupby('state_code')['state_name'].nunique() > 1).sum())
print("Rows that failed to map (NaN):", df["state_code"].isna().sum())

Codes mapping to >1 name: 0
Rows that failed to map (NaN): 0


In [50]:
print(violations)

      id       year  state_code   state_name  route_length  total_track
178  178  2017-2018           5  Uttarakhand           341          189


In [52]:
print(df[df['state_name'] == 'Uttarakhand'][['year', 'route_length', 'total_track']].to_string(index=False))

     year  route_length  total_track
2012-2013           345          556
2013-2014           345          518
2014-2015           345          518
2015-2016           340          509
2016-2017           340          465
2017-2018           341          189
2018-2019           341          527
2019-2020           346          528
2020-2021           346          555
2021-2022           346          517


In [53]:
uk = df[df["state_name"] == "Uttarakhand"]
before = uk.loc[uk["year"] == "2016-2017", "total_track"].values[0]
after  = uk.loc[uk["year"] == "2018-2019", "total_track"].values[0]
estimate = round((before + after) / 2)
print(f"Neighbors: {before} and {after}  ->  estimate = {estimate}")

Neighbors: 465 and 527  ->  estimate = 496


In [54]:
mask = (df["state_name"] == "Uttarakhand") & (df["year"] == "2017-2018")
df.loc[mask, "total_track"] = estimate

In [55]:
print("Invariant violations remaining:", (df["route_length"] > df["total_track"]).sum())
print(df[mask][["year", "state_name", "route_length", "total_track"]].to_string(index=False))

Invariant violations remaining: 0
     year  state_name  route_length  total_track
2017-2018 Uttarakhand           341          496


In [56]:
df["state_name"] = df["state_name"].replace({"Chhatisgarh": "Chhattisgarh"})

In [57]:
out = r"C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks-clean.csv"
df.to_csv(out, index=False)
print("\nSaved cleaned data to:", out)


Saved cleaned data to: C:\Users\v-vvajpai\Desktop\Project\Project_2\datasets\railway-networks-clean.csv
